# SSTW-noise G0: structured initial-noise saved-MP4 diagnostic

`DIAGNOSTIC_ONLY`. This exact4 run asks one question: does a frozen two-dimensional structured perturbation of the real Wan initial noise survive the unchanged 8-step generation, VAE decode, first H264 MP4, and a single-video fixed VAE-inversion observation? It does not use a matched OFF in the observer and does not tune after results.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import json, os, pathlib, shutil, subprocess, sys, zipfile, hashlib, queue, threading, time, traceback

REPOSITORY_URL = 'https://github.com/RICHAAARC/SC-SSTW-Feasibility.git'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
AUTHORIZED_REF = 'b7b4c7a343c5b2c0af736786821695fa6f351de0'
RUN_ID = '8ebdd045ca397a98'
AUTHORIZE_EXECUTION = True
AUTHORIZE_DRIVE_IO = True
WORK = pathlib.Path('/content/sstw-noise-g0-work')
RUN_ROOT = pathlib.Path(f'/content/sstw-noise-g0-{RUN_ID}')
OUTPUT = RUN_ROOT / 'output'
DRIVE_ROOT = pathlib.Path('/content/drive/MyDrive/SC-SSTW-Feasibility/sstw-noise-g0-structured-initial')
DRIVE_ARCHIVE = DRIVE_ROOT / f'sstw-noise-g0-{RUN_ID}.zip'
DRIVE_SIDECAR = DRIVE_ROOT / f'sstw-noise-g0-{RUN_ID}.zip.sha256.json'
if RUN_ROOT.exists(): shutil.rmtree(RUN_ROOT)
RUN_ROOT.mkdir(parents=True)
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
BOOTSTRAP_ERROR = None
resolved = AUTHORIZED_REF
runtime = {}
try:
    assert AUTHORIZE_EXECUTION and AUTHORIZE_DRIVE_IO
    import torch
    runtime = {'torch': torch.__version__, 'cuda': torch.version.cuda, 'cuda_available': torch.cuda.is_available(), 'bf16_supported': bool(torch.cuda.is_available() and torch.cuda.is_bf16_supported()), 'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None, 'vram_gib': round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2) if torch.cuda.is_available() else None}
    print('Runtime diagnostic:', runtime)
    if not runtime['cuda_available'] or not runtime['bf16_supported']: raise RuntimeError('CUDA and BF16 capabilities are required')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'diffusers==0.35.2', 'transformers', 'accelerate', 'ftfy', 'sentencepiece', 'safetensors', 'huggingface_hub', 'imageio', 'imageio-ffmpeg'])
    if WORK.exists(): shutil.rmtree(WORK)
    subprocess.check_call(['git', 'clone', '--filter=blob:none', REPOSITORY_URL, str(WORK)])
    subprocess.check_call(['git', '-C', str(WORK), 'checkout', '--detach', AUTHORIZED_REF])
    resolved = subprocess.check_output(['git', '-C', str(WORK), 'rev-parse', 'HEAD'], text=True).strip()
    if resolved != AUTHORIZED_REF: raise RuntimeError('authorized implementation ref mismatch')
except BaseException as exc:
    BOOTSTRAP_ERROR = {'type': type(exc).__name__, 'message': str(exc), 'traceback': traceback.format_exc()}
    print({'status': 'BOOTSTRAP_FAILED', **BOOTSTRAP_ERROR})
(RUN_ROOT / 'bootstrap_status.json').write_text(json.dumps({'runtime': runtime, 'error': BOOTSTRAP_ERROR}, sort_keys=True) + '\n')
print({'authorized_commit': resolved, 'run_id': RUN_ID, 'drive_archive': str(DRIVE_ARCHIVE)})

In [ ]:
combined_path = RUN_ROOT / 'runner.combined.txt'
argv = [sys.executable, str(WORK / 'experiments/run_g0_structured_initial_noise.py'), '--output', str(OUTPUT)]
return_code = None
report = None
orchestration_error = None
started = time.monotonic()
try:
    if BOOTSTRAP_ERROR is not None: raise RuntimeError('bootstrap failed; runner not started')
    stream_queue = queue.Queue()
    with combined_path.open('w') as log:
        process = subprocess.Popen(argv, cwd=WORK, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        def pump():
            try:
                for item in process.stdout: stream_queue.put(item)
            finally: stream_queue.put(None)
        threading.Thread(target=pump, daemon=True).start()
        while True:
            try: line = stream_queue.get(timeout=30)
            except queue.Empty:
                elapsed = time.monotonic() - started; alive = process.poll() is None
                print({'event': 'NOTEBOOK_HEARTBEAT', 'elapsed_seconds': round(elapsed, 1), 'runner_alive': alive}, flush=True)
                if elapsed > 3600 and alive:
                    process.terminate()
                    try: process.wait(timeout=15)
                    except subprocess.TimeoutExpired: process.kill(); process.wait()
                    raise RuntimeError('runner exceeded 60 minute diagnostic budget')
                if not alive: break
                continue
            if line is None: break
            print(line, end='', flush=True); log.write(line); log.flush()
            try:
                value = json.loads(line)
                if isinstance(value, dict) and 'status' in value: report = value
            except json.JSONDecodeError: pass
        return_code = process.wait()
    print('Runner return code:', return_code)
    if report is None: raise RuntimeError(f'runner emitted no final status JSON; return_code={return_code}')
    expected_codes = {'STRUCTURED_NOISE_PRIMITIVE_FEASIBLE': 0, 'STRUCTURED_NOISE_PRIMITIVE_NOT_FEASIBLE': 3, 'INSTRUMENTATION_INSUFFICIENT': 2}
    if report.get('status') not in expected_codes or return_code != expected_codes[report['status']]: raise RuntimeError('runner status/exit mismatch')
except BaseException as exc:
    orchestration_error = {'type': type(exc).__name__, 'message': str(exc), 'traceback': traceback.format_exc()}
    report = {'status': 'INSTRUMENTATION_INSUFFICIENT', 'diagnostic_class': 'DIAGNOSTIC_ONLY', 'reason': 'NOTEBOOK_ORCHESTRATION_FAILURE', 'message': str(exc)}
finally:
    status_record = {'report': report, 'return_code': return_code, 'bootstrap_error': BOOTSTRAP_ERROR, 'orchestration_error': orchestration_error, 'argv': argv, 'elapsed_seconds': round(time.monotonic() - started, 3)}
    (RUN_ROOT / 'notebook_status.json').write_text(json.dumps(status_record, sort_keys=True) + '\n')
    local_archive = pathlib.Path(f'/content/sstw-noise-g0-{RUN_ID}.zip')
    with zipfile.ZipFile(local_archive, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
        for path in sorted(RUN_ROOT.rglob('*')):
            if path.is_file(): archive.write(path, path.relative_to(RUN_ROOT.parent))
    archive_sha256 = hashlib.sha256(local_archive.read_bytes()).hexdigest()
    shutil.copy2(local_archive, DRIVE_ARCHIVE)
    sidecar = {'schema_version': 1, 'diagnostic_class': 'DIAGNOSTIC_ONLY', 'run_id': RUN_ID, 'authorized_ref': resolved, 'archive_sha256': archive_sha256, 'status': report['status'], 'return_code': return_code}
    DRIVE_SIDECAR.write_text(json.dumps(sidecar, sort_keys=True) + '\n')
    if hashlib.sha256(DRIVE_ARCHIVE.read_bytes()).hexdigest() != archive_sha256: raise RuntimeError('Drive ZIP copy failed')
summary = {**report, 'run_id': RUN_ID, 'archive_sha256': archive_sha256, 'drive_zip': str(DRIVE_ARCHIVE), 'drive_sidecar': str(DRIVE_SIDECAR)}
print(summary)
if report['status'] == 'INSTRUMENTATION_INSUFFICIENT': raise RuntimeError('structured-noise instrumentation insufficient; packaged diagnostics are available')